# TXPipe - Reduced Shear

This notebook illustrates the new option to compute **reduced shear** directly within TXPipe using CLMM.  
Instead of producing excess surface density (ΔΣ), the pipeline here outputs background galaxy shapes in terms of reduced shear, providing an alternative lensing observable for cluster analyses.  

We will walk through:  
- Running the TXPipe stages needed to build per-cluster reduced shear catalogs  
- Inspecting the structure of the outputs  
- Demonstrating how these catalogs can be used as CLMM inputs


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import clmm

print(clmm.__version__)
import pyccl as ccl

print(ccl.__version__)

1.14.1
3.2.1


In [3]:
import os
from pprint import pprint

import ceci

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from scipy.stats import binned_statistic

# 1 deg$^2$ Sample

We will do some runs on the 1 deg^2 example data set with around 80k galaxies. This is small enough that we can do it all in jupyter.

The data set, which is based on CosmoDC2, contains pre-computed photo-z and and contains a RedMapper cluster catalog for the field.

We will clone our own copy of the TXPipe directory, and run this notebook from there.  **Please change `my_txpipe_dir`** to your own version of the path when running this:

In [4]:
my_txpipe_dir = "/sps/lsst/users/ebarroso/TXPipe"  # "/sps/lsst/groups/clusters/cl_pipeline_project/TXPipe"
# my_txpipe_dir = "/sps/lsst/users/maguena/git_codes/TXPipe"
import sys

sys.path.append(
    my_txpipe_dir
)  # "/sps/lsst/groups/clusters/cl_pipeline_project/TXPipe")
sys.path.append(
    f"{my_txpipe_dir}/txpipe/extensions"
)  # "/sps/lsst/groups/clusters/cl_pipeline_project/TXPipe/txpipe/extensions/")
import importlib

import txpipe

Now we make an output directory for everything, if it doesn't exist already.

In [5]:
inputs_dir = f"{my_txpipe_dir}/data/example/inputs"
out_dir = f"{my_txpipe_dir}/data/example/outputs_shear_catalogs"
os.makedirs(out_dir, exist_ok=True)

# inputs
shear_catalog = f"{inputs_dir}/metadetect_shear_catalog.hdf5"
cluster_catalog = f"{inputs_dir}/cluster_catalog.hdf5"
source_photoz_pdfs = f"{inputs_dir}/photoz_pdfs.hdf5"
fiducial_cosmology = f"{my_txpipe_dir}/data/fiducial_cosmology.yml"

# outputs of the source sample stages and of the cluster binning
shear_selection = f"{out_dir}/shear_selection.hdf5"
shear_bin_assignments = f"{out_dir}/shear_bin_assignments.hdf5"
shear_tomography_catalog = f"{out_dir}/shear_tomography_catalog.hdf5"
cluster_catalog_tomography = f"{out_dir}/cluster_catalog_tomography.hdf5"

In [6]:
if not os.path.exists(
    f"{my_txpipe_dir}/data/example/inputs/metadetect_shear_catalog.hdf5"
):
    raise RuntimeError("Download and extract the sample data file to continue")

---

## WL sample selection

The source sample is made by three stages. `TXMetaDetectSelection` applies the S/N, size and flag cuts to each metadetect variant (`shear_selection`). `TXSimpleRedshiftSourceTomography` puts the objects in tomographic bins of true redshift (`shear_bin_assignments`). `TXComputeShearCalibrationMetadetect` computes the calibration of the selected objects in each bin and in the whole sample (`shear_tomography_catalog`).

The cluster stages use the selection and the bins. By default the sources must be in one of the bins. Their shears are calibrated with the response of the sources of each cluster stack (`CLClusterShearCalibrator`), not with the calibration of the whole sample.

In [7]:
step1 = txpipe.TXMetaDetectSelection.make_stage(
    # This file is the input metadetect shear catalog
    shear_catalog=shear_catalog,
    # This is the output file for this stage: whether each object of each variant passes the cuts
    shear_selection=shear_selection,
    output_dir=out_dir,
)
step1.run()
step1.finalize()

# tomographic bins from the true redshifts
step1b = txpipe.TXSimpleRedshiftSourceTomography.make_stage(
    shear_catalog=shear_catalog,
    shear_bin_assignments=shear_bin_assignments,
    output_dir=out_dir,
    redshift_col="redshift_true",
    source_zbin_edges=[0.1, 0.5, 1.0, 3.0],
    chunk_rows=100_000,
)
step1b.run()
step1b.finalize()

# calibration of the selected objects
step1c = txpipe.TXComputeShearCalibrationMetadetect.make_stage(
    shear_catalog=shear_catalog,
    shear_bin_assignments=shear_bin_assignments,
    shear_selection=shear_selection,
    shear_tomography_catalog=shear_tomography_catalog,
    output_dir=out_dir,
    delta_gamma=0.02,
)
step1c.run()
step1c.finalize()

/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:18: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  elif hasattr(module, "__version__"):
/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:19: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  v = module.__version__


Rank 0 processing rows 0 - 10,000
Rank 0 processing rows 10,000 - 20,000
Rank 0 processing rows 20,000 - 30,000
Rank 0 processing rows 30,000 - 40,000
Rank 0 processing rows 40,000 - 50,000
Rank 0 processing rows 50,000 - 60,000
Rank 0 processing rows 60,000 - 70,000
Rank 0 processing rows 70,000 - 80,000
Rank 0 processing rows 80,000 - 82,200
Rank 0 processing rows 0 - 10,000
Rank 0 processing rows 10,000 - 20,000
Rank 0 processing rows 20,000 - 30,000
Rank 0 processing rows 30,000 - 40,000
Rank 0 processing rows 40,000 - 50,000
Rank 0 processing rows 50,000 - 60,000
Rank 0 processing rows 60,000 - 70,000
Rank 0 processing rows 70,000 - 80,000
Rank 0 processing rows 80,000 - 82,200
Rank 0 processing rows 0 - 10,000
Rank 0 processing rows 10,000 - 20,000
Rank 0 processing rows 20,000 - 30,000
Rank 0 processing rows 30,000 - 40,000
Rank 0 processing rows 40,000 - 50,000
Rank 0 processing rows 50,000 - 60,000
Rank 0 processing rows 60,000 - 70,000
Rank 0 processing rows 70,000 - 80,000
R

/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:18: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  elif hasattr(module, "__version__"):
/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:19: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  v = module.__version__


/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:18: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  elif hasattr(module, "__version__"):
/sps/lsst/users/ebarroso/TXPipe/txpipe/utils/provenance.py:19: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  v = module.__version__


Process 0 running selection for rows 0-10,000
Process 0 running selection for rows 10,000-20,000
Process 0 running selection for rows 20,000-30,000
Process 0 running selection for rows 30,000-40,000
Process 0 running selection for rows 40,000-50,000
Process 0 running selection for rows 50,000-60,000
Process 0 running selection for rows 60,000-70,000
Process 0 running selection for rows 70,000-80,000
Process 0 running selection for rows 80,000-82,200
Process 0 running selection for rows 0-10,000
Process 0 running selection for rows 10,000-20,000
Process 0 running selection for rows 20,000-30,000
Process 0 running selection for rows 30,000-40,000
Process 0 running selection for rows 40,000-50,000
Process 0 running selection for rows 50,000-60,000


Process 0 running selection for rows 60,000-70,000
Process 0 running selection for rows 70,000-80,000
Process 0 running selection for rows 80,000-82,200
Process 0 running selection for rows 0-10,000
Process 0 running selection for rows 10,000-20,000
Process 0 running selection for rows 20,000-30,000
Process 0 running selection for rows 30,000-40,000
Process 0 running selection for rows 40,000-50,000
Process 0 running selection for rows 50,000-60,000
Process 0 running selection for rows 60,000-70,000
Process 0 running selection for rows 70,000-80,000
Process 0 running selection for rows 80,000-82,200
Process 0 running selection for rows 0-10,000
Process 0 running selection for rows 10,000-20,000
Process 0 running selection for rows 20,000-30,000
Process 0 running selection for rows 30,000-40,000
Process 0 running selection for rows 40,000-50,000
Process 0 running selection for rows 50,000-60,000
Process 0 running selection for rows 60,000-70,000
Process 0 running selection for rows 70,0

Process 0 running selection for rows 10,000-20,000
Process 0 running selection for rows 20,000-30,000
Process 0 running selection for rows 30,000-40,000
Process 0 running selection for rows 40,000-50,000
Process 0 running selection for rows 50,000-60,000
Process 0 running selection for rows 60,000-70,000
Process 0 running selection for rows 70,000-80,000
Process 0 running selection for rows 80,000-82,200


By default the cluster stages keep the objects that pass the selection cuts and are in a tomographic bin (option `require_tomo_bin`). Objects outside the bins have `bin = -1` in the tomography file:

In [8]:
import h5py

with h5py.File(shear_selection, "r") as f:
    selection = f["shear/ns/selection"][:]
with h5py.File(shear_tomography_catalog, "r") as f:
    tomo_bin = f["tomography/bin"][:]
print(
    f"{selection.size:,} objects, {selection.sum():,} pass the selection cuts, "
    f"{(tomo_bin >= 0).sum():,} of them are in a tomographic bin"
)

82,200 objects, 12,383 pass the selection cuts, 12,257 of them are in a tomographic bin


## Inspecting the output of the first stages

In [9]:
import h5py

with h5py.File(
    shear_tomography_catalog,
    "r",
) as f:
    # Print all root level object names (aka keys)
    print("Keys: %s" % f.keys())
    # Access the 'tomography' group and then the 'bin' dataset
    bin_data = f["tomography"]["bin"][()]
    print([bin_data[i] for i in range(len(bin_data)) if bin_data[i] != -1])
    # Count how many entries are not -1
    count = len(bin_data[bin_data != -1])
    print(count)

Keys: <KeysViewHDF5 ['counts', 'provenance', 'response', 'tomography']>
[np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.i

In [10]:
import h5py

with h5py.File(
    f"{my_txpipe_dir}/data/example/inputs/metadetect_shear_catalog.hdf5", "r"
) as f:
    # Print all root level object names (aka keys)
    # these can be group or dataset names
    print("Keys: %s" % f.keys())
    # get first object name/key; may or may NOT be a group
    print(f["shear"]["00"].keys())

Keys: <KeysViewHDF5 ['shear']>
<KeysViewHDF5 ['T', 'T_err', 'dec', 'flags', 'g1', 'g2', 'id', 'mag_err_i', 'mag_err_r', 'mag_err_z', 'mag_i', 'mag_r', 'mag_z', 'psf_T_mean', 'psf_g1', 'psf_g2', 'ra', 'redshift_true', 's2n', 'true_g1', 'true_g2', 'weight']>


---

## Cluster-source pairs

Our second step finds the sources behind every cluster. It saves only the pairs: for each cluster, the index of its sources in the shear catalog and their distance to the cluster. The shears, weights and profiles are computed later by the ensemble stage, so the options for delta sigma or reduced shear and for the weights now go to that stage.

In [ ]:
print("Options for this pipeline and their defaults:")
import txpipe.extensions as extensions

print(extensions.CLClusterSourceSelect.config_options)


# The photo-z file is only needed by the photo-z redshift criteria, otherwise it can be "none"
photoz_criteria = ["zmode", "zmean", "zmedian", "pdf"]


def run_selection(cut_criterion, name, use_radius=False):
    cluster_shear_selection = f"{out_dir}/my_cluster_shear_selection_{name}.hdf5"
    step2 = extensions.CLClusterSourceSelect.make_stage(
        # Shear catalog, as before
        shear_catalog=shear_catalog,
        # This is the initial cluster catalog - RAs, Decs, richess, redshift, etc.
        cluster_catalog=cluster_catalog,
        # This fiducial cosmology is used to convert the search radius in Mpc to an angle
        fiducial_cosmology=fiducial_cosmology,
        # Whether each object passes the cuts of the source selection
        shear_selection=shear_selection,
        # By default the sources must be in one of its tomographic bins (require_tomo_bin)
        shear_bin_assignments=shear_bin_assignments,
        # This is a QP file created by RAIL to generate the photo-zs for this sample
        source_photoz_pdfs=source_photoz_pdfs if cut_criterion in photoz_criteria else "none",
        # This is the output for this stage
        cluster_shear_selection=cluster_shear_selection,
        output_dir=out_dir,
        # Let's override some of the configuration parameters for this stage:
        max_radius=10.0,
        use_radius=use_radius,
        max_angle=40,
        redshift_cut_criterion=cut_criterion,
    )
    step2.run()
    step2.finalize()
    return cluster_shear_selection

In [ ]:
# For each case: the redshift used for the cut behind the cluster and for Sigma_crit,
# delta sigma or reduced shear, a search radius of 10 Mpc (or 40 arcmin), and the radial
# bins of the profile
cases = {
    "mean_ds": dict(cut="ztrue", weight="ztrue", delta_sigma=True, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
    "mode_ds": dict(cut="zmode", weight="zmode", delta_sigma=True, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
    "pdf_ds": dict(cut="pdf", weight="pdf", delta_sigma=True, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
    "mean_gamma": dict(cut="zmean", weight="zmean", delta_sigma=False, use_radius=False, units="arcmin", r_min=5.0, r_max=25.0),
    "mode_gamma": dict(cut="zmode", weight="zmode", delta_sigma=False, use_radius=False, units="arcmin", r_min=5.0, r_max=25.0),
    "pdf_gamma": dict(cut="pdf", weight="pdf", delta_sigma=False, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
    "mean_tomo_bins": dict(cut="tomo_bins", weight="ztrue", delta_sigma=False, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
    "mode_tomo_bins": dict(cut="tomo_bins", weight="zmode", delta_sigma=False, use_radius=True, units="mpc", r_min=0.2, r_max=3.0),
}

pair_files = {
    name: run_selection(case["cut"], name, use_radius=case["use_radius"])
    for name, case in cases.items()
}

## Binning Cluster Catalog

In [ ]:
print("Options for this pipeline and their defaults:")
print(extensions.CLClusterBinningRedshiftRichness.config_options)
step3 = extensions.CLClusterBinningRedshiftRichness.make_stage(
    cluster_catalog=cluster_catalog,
    richedge=[1, 80],
    zedge=[0.2, 1.0],
    cluster_catalog_tomography=cluster_catalog_tomography,
)
print(step3.config)
step3.run()
step3.finalize()

## Making the Ensemble Profile

This stage reads the sources of each cluster from the catalogs, calibrates their shears with the calibration of the whole source sample, computes the weights and the delta sigma or reduced shear, and stacks the profiles of the clusters of each bin. The type of profile is saved as `profile_type` in the output. The sources of each cluster, with these quantities, are saved in `cluster_source_quantities`.

In [ ]:
radius_bins = 5


def run_ensemble(name):
    case = cases[name]
    cluster_profiles = f"{out_dir}/cluster_profiles_{name}.pkl"
    # The response of the sources of each stack, in the radial bins of the profile
    cluster_shear_response = f"{out_dir}/cluster_shear_response_{name}.hdf5"
    step3 = extensions.CLClusterShearCalibrator.make_stage(
        cluster_catalog_tomography=cluster_catalog_tomography,
        cluster_shear_selection=pair_files[name],
        fiducial_cosmology=fiducial_cosmology,
        shear_catalog=shear_catalog,
        cluster_shear_response=cluster_shear_response,
        output_dir=out_dir,
        delta_gamma=0.02,
        bin_units=case["units"],
        r_min=case["r_min"],
        r_max=case["r_max"],
        nbins=radius_bins,
    )
    step3.run()
    step3.finalize()
    step4 = extensions.CLClusterEnsembleProfiles.make_stage(
        shear_catalog=shear_catalog,
        fiducial_cosmology=fiducial_cosmology,
        cluster_shear_selection=pair_files[name],
        cluster_catalog_tomography=cluster_catalog_tomography,
        source_photoz_pdfs=source_photoz_pdfs if case["weight"] in photoz_criteria else "none",
        output_dir=out_dir,
        cluster_profiles=cluster_profiles,
        cluster_source_quantities=f"{out_dir}/cluster_source_quantities_{name}.hdf5",
        delta_sigma=case["delta_sigma"],
        redshift_weight_criterion=case["weight"],
        shear_calibration="cluster_stack",
        cluster_shear_response=cluster_shear_response,
        bin_units=case["units"],
        r_min=case["r_min"],
        r_max=case["r_max"],
        nbins=radius_bins,
        cov_type="sample_covariance",
    )
    step4.run()
    step4.finalize()
    return cluster_profiles

In [ ]:
cluster_ensemble_files = {name: run_ensemble(name) for name in cases}

## Exploring the sources of each cluster

To avoid making lots and lots of copies of the data, the pairs stage has not made a catalog, but instead made an index into the other catalogs. The ensemble stage saves the sources of each cluster in the bins with the quantities it computed for them (calibrated shears, weights, tangential and cross components and, for delta sigma, Sigma_crit) in `cluster_source_quantities`.

We read that file directly: `catalog/cluster_sample_start` and `catalog/cluster_sample_count` give where the sources of each cluster are in the `index` group.

In [ ]:
import h5py
import pickle


def cluster_sources(name, i):
    """The sources of cluster i (row i of the catalog group) of case name, with their quantities from the ensemble stage."""
    with h5py.File(f"{out_dir}/cluster_source_quantities_{name}.hdf5", "r") as f:
        start = f["catalog/cluster_sample_start"][i]
        end = start + f["catalog/cluster_sample_count"][i]
        return {col: f[f"index/{col}"][start:end] for col in f["index"].keys()}

In [ ]:
bg_catalogs = {name: cluster_sources(name, 0) for name in cases}

In [ ]:
for name_groups in [
    [
        "mean_ds",
        "mode_ds",
        "pdf_ds",
    ],
    [
        "mean_gamma",
        "mode_gamma",
        "pdf_gamma",
    ],
    ["mean_tomo_bins", "mode_tomo_bins"],
]:
    for name in name_groups:
        plt.scatter(
            bg_catalogs[name]["distance_arcmin"],
            bg_catalogs[name]["tangential_comp"],
            label=name,
        )
    plt.ylabel("Tangential Shear Component")
    plt.xlabel("Distance from Cluster (arcmin)")
    plt.legend()
    plt.show()

In [ ]:
rbins = np.linspace(0, 40, 41)
for name, _dat in bg_catalogs.items():
    if "ds" in name:
        plt.plot(
            0.5 * (rbins[:-1] + rbins[1:]),
            binned_statistic(
                bg_catalogs[name]["distance_arcmin"],
                bg_catalogs[name]["tangential_comp"],
                bins=rbins,
            )[0],
            label=name,
        )
plt.ylabel("Delta Sigma")
plt.xlabel("Distance from Cluster (arcmin)")
plt.legend()
plt.show()

In [ ]:
rbins = np.linspace(0, 40, 41)
for name, _dat in bg_catalogs.items():
    if "ds" not in name:
        plt.plot(
            0.5 * (rbins[:-1] + rbins[1:]),
            binned_statistic(
                bg_catalogs[name]["distance_arcmin"],
                bg_catalogs[name]["tangential_comp"],
                bins=rbins,
            )[0],
            label=name,
        )
plt.ylabel("Tangential Shear Component")
plt.xlabel("Distance from Cluster (arcmin)")
plt.legend()
plt.show()

We can also look at one cluster: its sources on the sky, and its profile in the ensemble of its bin:

In [ ]:
name = "mode_ds"
with h5py.File(f"{out_dir}/cluster_source_quantities_{name}.hdf5", "r") as f:
    cluster_ids = f["catalog/cluster_id"][:]
    sample_count = f["catalog/cluster_sample_count"][:]
# the cluster with the most sources
i = int(np.argmax(sample_count))
with h5py.File(cluster_catalog, "r") as f:
    row = np.flatnonzero(f["clusters/cluster_id"][:] == cluster_ids[i])[0]
    info = {col: f[f"clusters/{col}"][row] for col in f["clusters"].keys()}
print(info)
sources = cluster_sources(name, i)
plt.scatter(sources["ra"], sources["dec"], c=sources["tangential_comp"], s=2)
plt.colorbar(label="Delta Sigma of each source")
plt.xlabel("ra")
plt.ylabel("dec")
plt.show()

with open(cluster_ensemble_files[name], "rb") as f:
    profiles = pickle.load(f)
for key, value in profiles.items():
    ensemble = value["clmm_cluster_ensemble"]
    if ensemble is not None and info["cluster_id"] in ensemble.data["cluster_id"]:
        print(key)
        print(ensemble.data[ensemble.data["cluster_id"] == info["cluster_id"]])

## SACC File
Lastly we are running the SACC stage to show the profiles. THis stage has to be addapted for the reduced shear as we can measure it in angles instead of physical radii. Also, we need to decide on the type of data that we will assign to the different options.

In [ ]:
print("Options for this pipeline and their defaults:")
print(txpipe.extensions.CLClusterSACC.config_options)


def run_sacc(name, cluster_profiles):
    cluster_sacc_catalog = f"{out_dir}/cluster_sacc_catalog_{name}.sacc"
    step5 = txpipe.extensions.CLClusterSACC.make_stage(
        cluster_profiles=cluster_profiles,
        survey_name="cosmodc2-1deg",
        area=1.0,
        # the ensembles have the sample covariance
        cov_component="tan_sc",
        cluster_sacc_catalog=cluster_sacc_catalog,
    )
    step5.run()
    step5.finalize()
    return cluster_sacc_catalog

In [ ]:
import sacc

print(sacc.__version__)

In [ ]:
test_sacc = {name: run_sacc(name, path) for name, path in cluster_ensemble_files.items()}

### Here we plot the profile of one bin for each file

In [ ]:
import sacc

for name, path in test_sacc.items():
    t = sacc.Sacc.load_fits(path)
    print(name, t.tracers)
    if cases[name]["delta_sigma"]:
        data_type = sacc.data_types.standard_types.cluster_delta_sigma
    else:
        data_type = sacc.data_types.standard_types.cluster_shear
    data2 = []
    radius2 = []
    for i in range(0, radius_bins):
        trac = ("cosmodc2-1deg", "bin_rich_0", "bin_z_0", f"radius_{i}")
        print(t.get_data_points(data_type, trac))
        data2.append(t.get_data_points(data_type, trac)[0].value)
        radius2.append(t.tracers[f"radius_{i}"].center)
    plt.plot(radius2, data2, label=name)
    plt.ylabel("shear profile")
    plt.xlabel("Radial bin")
    plt.legend()
    plt.show()


for name, path in test_sacc.items():
    t = sacc.Sacc.load_fits(path)
    print(name, t.tracers)
    data_type = sacc.data_types.standard_types.cluster_counts
    print(t.data)